# Phase 1: Data Preparation

This notebook downloads, samples, cleans, and merges all the datasets required for training our B.Tech CSE AI/ML Tutor model.

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repository to get our processing scripts
!git clone https://github.com/YOUR_USERNAME/btech-ai-tutor.git
%cd btech-ai-tutor
!pip install -r requirements.txt

In [ ]:
# Run download script
!python scripts/download_datasets.py --output_dir data/raw

In [ ]:
# Run custom synthetic data generation (you must provide your Gemini API key)
# Set your GEMINI_API_KEY environment variable or pass it to the script
API_KEY = "YOUR_GEMINI_API_KEY"

!python scripts/generate_custom_data.py --api_key {API_KEY} --dataset_type cs_fundamentals --output_file data/custom/cs_fundamentals.jsonl --num_samples 200
!python scripts/generate_custom_data.py --api_key {API_KEY} --dataset_type ml_dl_concepts --output_file data/custom/ml_dl_concepts.jsonl --num_samples 200
!python scripts/generate_custom_data.py --api_key {API_KEY} --dataset_type advanced_ai_topics --output_file data/custom/advanced_ai_topics.jsonl --num_samples 100

# Repeat for Layer 4 datasets (Gate prep, interview prep, etc.)
!python scripts/generate_custom_data.py --api_key {API_KEY} --dataset_type gate_prep --output_file data/custom/gate_cs_prep.jsonl --num_samples 100
!python scripts/generate_custom_data.py --api_key {API_KEY} --dataset_type ml_interview --output_file data/custom/ml_interview_qa.jsonl --num_samples 100
!python scripts/generate_custom_data.py --api_key {API_KEY} --dataset_type dsa_interview --output_file data/custom/dsa_interview.jsonl --num_samples 100
!python scripts/generate_custom_data.py --api_key {API_KEY} --dataset_type system_design_ml --output_file data/custom/system_design_ml.jsonl --num_samples 50
!python scripts/generate_custom_data.py --api_key {API_KEY} --dataset_type viva_qa --output_file data/custom/viva_qa.jsonl --num_samples 50

In [ ]:
# Run DPO preference pair generation
!python scripts/create_dpo_pairs.py --api_key {API_KEY} --output_file data/custom/dpo_preferences.jsonl --num_samples 50

In [ ]:
# Run quality filter on custom datasets
!python scripts/quality_filter.py --input_dir data/custom --output_dir data/custom_filtered

In [ ]:
# Move filtered custom files back to custom/
!rm -rf data/custom && mv data/custom_filtered data/custom

In [ ]:
# Format and merge all datasets into final training files
!python scripts/format_and_merge.py --raw_dir data/raw --custom_dir data/custom --output_dir data/processed

In [ ]:
# Copy processed datasets to Google Drive for persistent access in subsequent training phases
!mkdir -p /content/drive/MyDrive/btech-ai-tutor/data
!cp -r data/processed/* /content/drive/MyDrive/btech-ai-tutor/data/
print("Datasets backed up to Google Drive!")